**Starting From Here is the final one**

In [ ]:
from sklearn.multioutput import MultiOutputRegressor
from lightgbm import LGBMRegressor
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
import gc
import torch
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_data_path = '/kaggle/input/datasets/dhammaruciwl/data-mining-final-project/data/train.csv'
test_data_path = '/kaggle/input/datasets/dhammaruciwl/data-mining-final-project/data/test.csv'

print("1. Loading raw datasets...")
train_raw = pd.read_csv(train_data_path)

# Parse dates
train_date_parts = train_raw['date'].str.split('-', expand=True)
train_raw['year'] = train_date_parts[0].astype(int)
train_raw['month'] = train_date_parts[1].astype(int)
train_raw['day'] = train_date_parts[2].astype(int)

region_score_counts = pd.crosstab(
    train_raw["region_id"],
    train_raw["score"]
)

region_score_counts

Using device: cuda
1. Loading raw datasets...


score,0.0,1.0,2.0,3.0,4.0,5.0
region_id,,,,,,
R1,394,197,77,57,27,30
R1001,569,101,59,27,19,7
R1002,619,71,62,26,4,0
R1005,607,101,41,22,11,0
R1006,630,65,63,20,4,0
...,...,...,...,...,...,...
R989,615,94,39,18,16,0
R990,603,95,40,36,8,0
R992,607,85,39,41,10,0


In [ ]:
train_raw['score'].value_counts()

score
0.0    1048333
1.0     303432
2.0     186279
3.0     118496
4.0      69422
5.0      31974
Name: count, dtype: int64

In [2]:
(region_score_counts == 0).sum()

score
0.0       0
1.0       0
2.0       4
3.0     138
4.0     406
5.0    1201
dtype: int64

In [4]:
month_score_counts = pd.crosstab(
    train_raw["month"],
    train_raw["score"]
)

month_score_counts

score,0.0,1.0,2.0,3.0,4.0,5.0
month,,,,,,
1,88204,26188,17289,10044,6434,2457
2,80673,22664,14941,9381,5209,2012
3,90216,26232,16758,10197,5568,1645
4,84504,25073,15461,9232,5629,1725
5,96631,25197,14386,8703,5633,2314
6,91021,22924,14173,8370,5036,2348
7,86490,24068,16522,11752,6638,2898
8,85635,26126,15928,12242,7047,3638
9,82687,26075,15420,10413,5985,3292


In [5]:
day_score_counts = pd.crosstab(
    train_raw["day"],
    train_raw["score"]
)

day_score_counts

score,0.0,1.0,2.0,3.0,4.0,5.0
day,,,,,,
1,35302,9076,6498,4152,2429,991
2,32946,9356,5265,3334,1896,1155
3,35742,11166,6786,3989,2389,624
4,32270,9695,6381,4149,2330,1375
5,33593,9796,5770,3773,2125,1143
6,38257,10500,6242,4113,2674,1158
7,33679,9536,6049,3861,2213,862
8,35354,9192,6480,3991,2409,1022
9,33197,9401,5021,3296,1911,1126


In [14]:
region_monthly_score_counts = pd.crosstab(
    index = [train_raw["region_id"], train_raw["month"]],
    columns = train_raw["score"]
)

display(region_monthly_score_counts.head(60))

score            0.0  1.0  2.0  3.0  4.0  5.0
region_id month                              
R1        1       28   23   11    0    0    5
          2       31   21    4    0    1    3
          3       35   24    4    0    4    0
          4       37   17    4    4    1    0
          5       40   14    5    7    2    0
          6       40    8   10    2    4    0
          7       38    5   11   10    1    1
          8       32   14    4   12    1    4
          9       27   20    5    5    3    4
          10      27   18    6    8    2    5
          11      26   21    4    3    7    4
          12      33   12    9    6    1    4
R1001     1       48   10    9    0    0    0
          2       52    1    7    0    0    0
          3       64    3    0    0    0    0
          4       59    4    0    0    0    0
          5       59    5    1    3    0    0
          6       53    3    4    2    2    0
          7       46   14    0    1    3    2
          8       39   13    7    2    2    4
          9       31   15   10    0    7    1
          10      35   12    9    5    5    0
          11      42    7    4   12    0    0
          12      41   14    8    2    0    0
R1002     1       49    9    9    0    0    0
          2       51    2    7    0    0    0
          3       62    5    0    0    0    0
          4       60    3    0    0    0    0
          5       65    3    0    0    0    0
          6       56    3    2    3    0    0
          7       49    7    5    5    0    0
          8       50    6    6    5    0    0
          9       45   12    3    3    1    0
          10      44   12    6    1    3    0
          11      42    5   11    7    0    0
          12      46    4   13    2    0    0
R1005     1       55    8    4    0    0    0
          2       51    3    6    0    0    0
          3       62    5    0    0    0    0
          4       59    4    0    0    0    0
          5       60    8    0    0    0    0
          6       52    6    5    1    0    0
          7       54    6    1    4    1    0
          8       48   11    1    5    2    0
          9       39   11    6    7    1    0
          10      39   13    8    3    3    0
          11      42   13    5    1    4    0
          12      46   13    5    1    0    0
R1006     1       50    8    9    0    0    0
          2       51    2    7    0    0    0
          3       61    6    0    0    0    0
          4       58    5    0    0    0    0
          5       65    3    0    0    0    0
          6       56    4    2    2    0    0
          7       50    7    7    2    0    0
          8       50   13    1    3    0    0
          9       48    7    5    3    1    0
          10      49    5    8    1    3    0
          11      45    2   11    7    0    0
          12      47    3   13    2    0    0

In [ ]:
# Varying window from 1,2,4,7,12 week (original baseline)
from sklearn.multioutput import MultiOutputRegressor
from lightgbm import LGBMRegressor
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
import gc
import torch
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("1. Loading raw datasets...")
train_raw = pd.read_csv(train_data_path)

# Parse dates
train_date_parts = train_raw['date'].str.split('-', expand=True)
train_raw['month'] = train_date_parts[1].astype(int)
train_raw['day'] = train_date_parts[2].astype(int)

train_raw = train_raw.sort_values(['region_id', 'date']).reset_index(drop=True)
train_raw['evap_stress'] = train_raw['tmp_max'] * (100 - train_raw['humidity'])

# Climate Normals Baselines
cols_to_baseline = ['prec', 'tmp', 'humidity', 'wind', 'surf_tmp', 'evap_stress']
climate_normals = train_raw.groupby(['region_id', 'month'])[cols_to_baseline].mean().reset_index()
climate_normals = climate_normals.rename(columns={c: f'normal_{c}' for c in cols_to_baseline})

train_raw = train_raw.merge(climate_normals, on=['region_id', 'month'], how='left')

# Targets
targets = ['target_w1', 'target_w2', 'target_w3', 'target_w4', 'target_w5']
for i, t in enumerate(targets):
    train_raw[t] = train_raw.groupby('region_id')['score'].shift(-(i+1)*7)

# SAFE WINDOWS  
print("Engineering capped rolling windows...")
weather_cols = ['prec', 'tmp', 'humidity', 'wind', 'surf_tmp', 'evap_stress']
windows = [7, 14, 28, 49, 84]

for w in windows:
    for col in weather_cols:
        train_raw[f'{col}_roll_mean_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).mean())
        train_raw[f'{col}_roll_max_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).max())

# Anomalies
print("Engineering regional anomalies...")
for w in [7, 14, 28, 49, 84]:
    for col in ['prec', 'tmp', 'humidity', 'wind', 'surf_tmp', 'evap_stress']:
        train_raw[f'{col}_anomaly_{w}'] = train_raw[f'{col}_roll_mean_{w}'] - train_raw[f'normal_{col}']
        
train_final = train_raw.dropna(subset=targets + ['score']).copy()

# Convert the region_id from a text string to a Pandas 'category' type
train_final['region_id'] = train_final['region_id'].astype('category')

# Calculate the historical average score for each region
region_avg_score = train_final.groupby('region_id')['score'].mean().reset_index().rename(columns={'score': 'historical_region_score'})

# Merge this back into your main dataframe
train_final = train_final.merge(region_avg_score, on='region_id', how='left')

print(f"\nTotal 7th-day valid labeled rows available for Model 1: {len(train_final)}")
print("  FULL DATASET RETRAINING  ")

print("1. Loading raw datasets...")
test_raw = pd.read_csv(test_data_path)

# Parse dates
test_date_parts = test_raw['date'].str.split('-', expand=True)
test_raw['month'] = test_date_parts[1].astype(int)
test_raw['day'] = test_date_parts[2].astype(int)

test_raw = test_raw.sort_values(['region_id', 'date']).reset_index(drop=True)
test_raw['evap_stress'] = test_raw['tmp_max'] * (100 - test_raw['humidity'])

# Climate Normals Baselines
test_raw = test_raw.merge(climate_normals, on=['region_id', 'month'], how='left')

# Targets
targets = ['target_w1', 'target_w2', 'target_w3', 'target_w4', 'target_w5']

# SAFE WINDOWS  
print("Engineering capped rolling windows...")
weather_cols = ['prec', 'tmp', 'humidity', 'wind', 'surf_tmp', 'evap_stress']
windows = [7, 14, 28, 49, 84]

for w in windows:
    for col in weather_cols:
        test_raw[f'{col}_roll_mean_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).mean())
        test_raw[f'{col}_roll_max_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).max())

# Anomalies
print("Engineering regional anomalies...")
for w in [7, 14, 28, 49, 84]:
    for col in ['prec', 'tmp', 'humidity', 'wind', 'surf_tmp', 'evap_stress']:
        test_raw[f'{col}_anomaly_{w}'] = test_raw[f'{col}_roll_mean_{w}'] - test_raw[f'normal_{col}']

features = [c for c in train_raw.columns if c not in ['region_id', 'date']]
# Convert the region_id from a text string to a Pandas 'category' type
test_raw['region_id'] = test_raw['region_id'].astype('category')

# Merge this back into your main dataframe
test_raw = test_raw.merge(region_avg_score, on='region_id', how='left')
test_final = test_raw.copy()

feature_names = [c for c in train_final.columns if c not in ['region_id', 'date', 'score', 'year'] + targets]

X = train_final[feature_names].fillna(0).values
y = train_final[targets].values

# 1.  use the FULL dataset (X, y, and global_sample_weights)
print(f"Training Final Master Model on all {len(X)} rows...")

master_model = MultiOutputRegressor(LGBMRegressor(
    objective='mae', n_estimators=1000, num_leaves=95, learning_rate=0.015, 
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, device='gpu'
)).fit(X, y)

# 2. Prepare the Test Set
test_submission_rows = test_final.groupby('region_id').last().reset_index()

X_test = test_submission_rows[feature_names].fillna(0).values

# 3. Predict the future 5 weeks 
print("Generating Final 5-Week Forecasts...")
final_predictions = np.clip(master_model.predict(X_test), 0, 5)

# 4. Format the output exactly as Kaggle demands
submission = pd.DataFrame({
    'region_id': test_submission_rows['region_id'],
    'pred_week1': final_predictions[:, 0],
    'pred_week2': final_predictions[:, 1],
    'pred_week3': final_predictions[:, 2],
    'pred_week4': final_predictions[:, 3],
    'pred_week5': final_predictions[:, 4]
})

# Save the file
submission.to_csv('submission_70_window_region.csv', index=False)
print("\nSuccess  'submission_weighted_master.csv' is saved and ready to upload to Kaggle.")

Using device: cuda
1. Loading raw datasets...
Engineering capped rolling windows...
Engineering regional anomalies...

Total 7th-day valid labeled rows available for Model 1: 1746696
--- FULL DATASET RETRAINING ---
1. Loading raw datasets...
Engineering capped rolling windows...
Engineering regional anomalies...
Training Final Master Model on all 1746696 rows...
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 28605
[LightGBM] [Info] Number of data points in the train set: 1746696, number of used features: 114
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 114 dense feature groups (193.23 MB) transferred to GPU in 0.120189 secs. 0 sparse feature groups
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 28605
[LightGBM] [Info] Number of data points in the train set: 1746696, number of used features: 114
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 114 dense feature groups (193.23 MB) transferred to GPU in 0.120616 secs. 0 sparse feature groups
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 28605
[LightGBM] [Info] Number of data points in the train set: 1746696, number of used features: 114
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Com

In [ ]:
# Varying window from 1,2,4,7,12 week + std measurement
from sklearn.multioutput import MultiOutputRegressor
from lightgbm import LGBMRegressor
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
import gc
import torch
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("1. Loading raw datasets...")
train_raw = pd.read_csv(train_data_path)

# Parse dates
train_date_parts = train_raw['date'].str.split('-', expand=True)
train_raw['year'] = train_date_parts[0].astype(int)
train_raw['month'] = train_date_parts[1].astype(int)
train_raw['day'] = train_date_parts[2].astype(int)

train_raw = train_raw.sort_values(['region_id', 'date']).reset_index(drop=True)
train_raw['evap_stress'] = train_raw['tmp_max'] * (100 - train_raw['humidity'])

# Climate Normals Baselines
cols_to_baseline = ['prec', 'tmp', 'humidity', 'wind', 'surf_tmp', 'evap_stress']
climate_normals = train_raw.groupby(['region_id', 'month'])[cols_to_baseline].mean().reset_index()
climate_normals = climate_normals.rename(columns={c: f'normal_{c}' for c in cols_to_baseline})

train_raw = train_raw.merge(climate_normals, on=['region_id', 'month'], how='left')

# Targets
targets = ['target_w1', 'target_w2', 'target_w3', 'target_w4', 'target_w5']
for i, t in enumerate(targets):
    train_raw[t] = train_raw.groupby('region_id')['score'].shift(-(i+1)*7)

# SAFE WINDOWS  
print("Engineering capped rolling windows...")
weather_cols = ['prec', 'tmp', 'humidity', 'wind', 'surf_tmp', 'evap_stress']
windows = [7, 14, 28, 49, 84]

for w in windows:
    for col in weather_cols:
        train_raw[f'{col}_roll_mean_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).mean())
        train_raw[f'{col}_roll_max_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).max())
        
        if w >= 28:
            train_raw[f'{col}_roll_std_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).std())

# Anomalies
print("Engineering regional anomalies...")
for w in [7, 14, 28, 49, 84]:
    for col in ['prec', 'tmp', 'humidity', 'wind', 'surf_tmp', 'evap_stress']:
        train_raw[f'{col}_anomaly_{w}'] = train_raw[f'{col}_roll_mean_{w}'] - train_raw[f'normal_{col}']
        
train_final = train_raw.dropna(subset=targets + ['score']).copy()

# Convert the region_id from a text string to a Pandas 'category' type
train_final['region_id'] = train_final['region_id'].astype('category')

# Calculate the historical average score for each region
region_avg_score = train_final.groupby(['region_id', 'month'])['score'].mean().reset_index().rename(columns={'score': 'month_region_score'})

# Merge this back into your main dataframe
train_final = train_final.merge(region_avg_score, on=['region_id', 'month'], how='left')

MONTH_LEN = {
    1: 31, 2: 28, 3: 31, 4: 30, 5: 31, 6: 30,
    7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31
}

def is_leap_year(y):
    return (y % 4 == 0 and y % 100  = 0) or (y % 400 == 0)

def add_days_get_future_month(year, month, day, add_days):
    y, m, d = int(year), int(month), int(day)
    
    while add_days > 0:
        month_len = 29 if (m == 2 and is_leap_year(y)) else MONTH_LEN[m]
        days_left_in_month = month_len - d
        
        if add_days <= days_left_in_month:
            d += add_days
            add_days = 0
        else:
            add_days -= (days_left_in_month + 1)
            d = 1
            m += 1
            if m > 12:
                m = 1
                y += 1
    
    return m
    
def add_future_month_features(df, region_avg_score, horizons=(1, 2, 3, 4, 5)):
    df = df.copy()
    
    for h in horizons:
        add_days = 7 * h
        df[f'future_month_w{h}'] = [
            add_days_get_future_month(y, m, d, add_days)
            for y, m, d in zip(df['year'], df['month'], df['day'])
        ]
        
        tmp = df[['region_id', f'future_month_w{h}']].rename(
            columns={f'future_month_w{h}': 'month'}
        )
        
        tmp = tmp.merge(
            region_avg_score,
            on=['region_id', 'month'],
            how='left'
        )

        df[f'future_region_month_score_w{h}'] = tmp['month_region_score'].values

    return df

train_final = add_future_month_features(train_final, region_avg_score)

print(f"\nTotal 7th-day valid labeled rows available for Model 1: {len(train_final)}")
print("  FULL DATASET RETRAINING  ")
print("1. Loading raw datasets...")
test_raw = pd.read_csv(test_data_path)

# Parse dates
test_date_parts = test_raw['date'].str.split('-', expand=True)
test_raw['year'] = test_date_parts[0].astype(int)
test_raw['month'] = test_date_parts[1].astype(int)
test_raw['day'] = test_date_parts[2].astype(int)

test_raw = test_raw.sort_values(['region_id', 'date']).reset_index(drop=True)
test_raw['evap_stress'] = test_raw['tmp_max'] * (100 - test_raw['humidity'])

# Climate Normals Baselines
test_raw = test_raw.merge(climate_normals, on=['region_id', 'month'], how='left')

# Targets
targets = ['target_w1', 'target_w2', 'target_w3', 'target_w4', 'target_w5']

# SAFE WINDOWS  
print("Engineering capped rolling windows...")
weather_cols = ['prec', 'tmp', 'humidity', 'wind', 'surf_tmp', 'evap_stress']
windows = [7, 14, 28, 49, 84]

for w in windows:
    for col in weather_cols:
        test_raw[f'{col}_roll_mean_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).mean())
        test_raw[f'{col}_roll_max_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).max())

        if w >= 28:
            test_raw[f'{col}_roll_std_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).std())

# Anomalies
print("Engineering regional anomalies...")
for w in [7, 14, 28, 49, 84]:
    for col in ['prec', 'tmp', 'humidity', 'wind', 'surf_tmp', 'evap_stress']:
        test_raw[f'{col}_anomaly_{w}'] = test_raw[f'{col}_roll_mean_{w}'] - test_raw[f'normal_{col}']

# Convert the region_id from a text string to a Pandas 'category' type
test_raw['region_id'] = test_raw['region_id'].astype('category')

# Merge this back into your main dataframe
test_raw = test_raw.merge(region_avg_score, on=['region_id', 'month'], how='left')
test_final = test_raw.copy()

test_final = add_future_month_features(test_final, region_avg_score)

feature_names = [c for c in train_final.columns if c not in ['region_id', 'date', 'score', 'year'] + targets]

X = train_final[feature_names].copy()
y = train_final[targets].values

# 1.  use the FULL dataset (X, y, and global_sample_weights)
print(f"Training Final Master Model on all {len(X)} rows...")

master_model = MultiOutputRegressor(LGBMRegressor(
    objective='mae', n_estimators=1000, num_leaves=95, learning_rate=0.015, 
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, device='gpu'
)).fit(X, y)

test_submission_rows = test_final.groupby('region_id').last().reset_index()
X_test = test_submission_rows[feature_names].copy()

# 3. Predict the future 5 weeks 
print("Generating Final 5-Week Forecasts...")

final_predictions = np.clip(master_model.predict(X_test), 0, 5)

# 3.a Find the nearest whole number for every single prediction
rounded_preds = np.floor(final_predictions)

# 3.b Calculate the absolute distance between the continuous prediction and the whole number
distance_to_int = np.abs(final_predictions - rounded_preds)

# 3.c If the distance is less than 0.20, snap to the whole number. Otherwise, keep the decimal.
final_predictions = np.where(distance_to_int < 0.20, rounded_preds, final_predictions)

# 4. Format the output exactly as Kaggle demands
submission = pd.DataFrame({
    'region_id': test_submission_rows['region_id'],
    'pred_week1': final_predictions[:, 0],
    'pred_week2': final_predictions[:, 1],
    'pred_week3': final_predictions[:, 2],
    'pred_week4': final_predictions[:, 3],
    'pred_week5': final_predictions[:, 4]
})

# Save the file
submission.to_csv('submission_70_window_region_monthly.csv', index=False)
print("\nSuccess  'submission_70_window_region_monthly.csv' is saved and ready to upload to Kaggle.")

Using device: cuda
1. Loading raw datasets...
Engineering capped rolling windows...
Engineering regional anomalies...

Total 7th-day valid labeled rows available for Model 1: 1746696
--- FULL DATASET RETRAINING ---
1. Loading raw datasets...
Engineering capped rolling windows...
Engineering regional anomalies...
Training Final Master Model on all 1746696 rows...
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 34535
[LightGBM] [Info] Number of data points in the train set: 1746696, number of used features: 142
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 142 dense feature groups (239.87 MB) transferred to GPU in 0.148714 secs. 0 sparse feature groups
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 34535
[LightGBM] [Info] Number of data points in the train set: 1746696, number of used features: 142
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 142 dense feature groups (239.87 MB) transferred to GPU in 0.147180 secs. 0 sparse feature groups
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 34535
[LightGBM] [Info] Number of data points in the train set: 1746696, number of used features: 142
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Com

In [ ]:
final_predictions = np.clip(master_model.predict(X_test), 0, 5)

# 3.a Find the nearest whole number for every single prediction
rounded_preds = np.floor(final_predictions)

# 3.b Calculate the absolute distance between the continuous prediction and the whole number
distance_to_int = np.abs(final_predictions - rounded_preds)

# 3.c If the distance is less than 0.20, snap to the whole number. Otherwise, keep the decimal.
final_predictions = np.where(distance_to_int < 0.20, rounded_preds, final_predictions)

# 3.a Find the nearest whole number for every single prediction
rounded_preds = np.round(final_predictions)

# 3.b Calculate the absolute distance between the continuous prediction and the whole number
distance_to_int = np.abs(final_predictions - rounded_preds)

# 3.c If the distance is less than 0.20, snap to the whole number. Otherwise, keep the decimal.
final_predictions = np.where(distance_to_int < 0.20, rounded_preds, final_predictions)

# 4. Format the output exactly as Kaggle demands
submission = pd.DataFrame({
    'region_id': test_submission_rows['region_id'],
    'pred_week1': final_predictions[:, 0],
    'pred_week2': final_predictions[:, 1],
    'pred_week3': final_predictions[:, 2],
    'pred_week4': final_predictions[:, 3],
    'pred_week5': final_predictions[:, 4]
})

# Save the file
submission.to_csv('submission_70_window_region_monthly_down_20_up_10.csv', index=False)
print("\nSuccess  'submission_70_window_region_monthly.csv' is saved and ready to upload to Kaggle.")


Success! 'submission_70_window_region_monthly.csv' is saved and ready to upload to Kaggle.


In [ ]:
# 5 weeks window
from sklearn.multioutput import MultiOutputRegressor
from lightgbm import LGBMRegressor
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
import gc
import torch
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("1. Loading raw datasets...")
train_raw = pd.read_csv(train_data_path)

# Parse dates
train_date_parts = train_raw['date'].str.split('-', expand=True)
train_raw['year'] = train_date_parts[0].astype(int)
train_raw['month'] = train_date_parts[1].astype(int)
train_raw['day'] = train_date_parts[2].astype(int)

train_raw = train_raw.sort_values(['region_id', 'date']).reset_index(drop=True)

# Targets
targets = ['target_w1', 'target_w2', 'target_w3', 'target_w4', 'target_w5']
for i, t in enumerate(targets):
    train_raw[t] = train_raw.groupby('region_id')['score'].shift(-(i+1)*7)

train_raw['evap_stress'] = train_raw['tmp_max'] * (100 - train_raw['humidity'])

# Climate Normals Baselines
weather_cols = ['prec', 'humidity', 'wind', 'surf_tmp', 'evap_stress', 'surf_pre', 'dp_tmp', 'tmp']
climate_normals_region_month = train_raw.groupby(['region_id', 'month'])[weather_cols].mean().reset_index()
climate_normals_region_month = climate_normals_region_month.rename(columns={c: f'normal_{c}' for c in weather_cols})

train_raw = train_raw.merge(climate_normals_region_month, on=['region_id', 'month'], how='left')

# SAFE WINDOWS  
print("Engineering capped rolling windows...")
windows = [7, 14, 21, 28, 35]

for w in windows:
    for col in weather_cols:
        train_raw[f'{col}_roll_mean_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).mean())
        train_raw[f'{col}_roll_max_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).max())
        if w >= 35:
            train_raw[f'{col}_roll_std_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).std())

# Anomalies
print("Engineering regional anomalies...")
for w in windows:
    for col in weather_cols:
        train_raw[f'{col}_anomaly_{w}'] = train_raw[f'{col}_roll_mean_{w}'] - train_raw[f'normal_{col}']
        
train_final = train_raw.dropna().copy()

# Convert the region_id from a text string to a Pandas 'category' type
train_final['region_id'] = train_final['region_id'].astype('category')

# Calculate the historical average score for each region
region_avg_score = train_final.groupby('region_id')['score'].mean().reset_index().rename(columns={'score': 'historical_region_score'})

# Merge this back into your main dataframe
train_final = train_final.merge(region_avg_score, on='region_id', how='left')

# Monthly region average score
month_region_avg_score = train_final.groupby(['region_id', 'month'])['score'].mean().reset_index().rename(columns={'score': 'month_region_score'})
# Merge this back into your main dataframe
train_final = train_final.merge(month_region_avg_score, on=['region_id', 'month'], how='left')

print(f"\nTotal 7th-day valid labeled rows available for Model 1: {len(train_final)}")

MONTH_LEN = {
    1: 31, 2: 28, 3: 31, 4: 30, 5: 31, 6: 30,
    7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31
}

def is_leap_year(y):
    return (y % 4 == 0 and y % 100  = 0) or (y % 400 == 0)

def add_days_get_future_month(year, month, day, add_days):
    y, m, d = int(year), int(month), int(day)
    
    while add_days > 0:
        month_len = 29 if (m == 2 and is_leap_year(y)) else MONTH_LEN[m]
        days_left_in_month = month_len - d
        
        if add_days <= days_left_in_month:
            d += add_days
            add_days = 0
        else:
            add_days -= (days_left_in_month + 1)
            d = 1
            m += 1
            if m > 12:
                m = 1
                y += 1
    
    return m
    
def add_future_month_features(df, region_avg_score, horizons=(1, 2, 3, 4, 5)):
    df = df.copy()
    
    for h in horizons:
        add_days = 7 * h
        df[f'future_month_w{h}'] = [
            add_days_get_future_month(y, m, d, add_days)
            for y, m, d in zip(df['year'], df['month'], df['day'])
        ]
        
        tmp = df[['region_id', f'future_month_w{h}']].rename(
            columns={f'future_month_w{h}': 'month'}
        )
        
        tmp = tmp.merge(
            region_avg_score,
            on=['region_id', 'month'],
            how='left'
        )

        df[f'future_region_month_score_w{h}'] = tmp['month_region_score'].values

    return df

train_final = add_future_month_features(train_final, month_region_avg_score)

feature_names = [c for c in train_final.columns if c not in ['region_id', 'date', 'score', 'year'] + targets]

X = train_final[feature_names]
y = train_final[targets].values

# 1.  use the FULL dataset (X, y, and global_sample_weights)
print(f"Training Final Master Model on all {len(X)} rows...")

master_model = MultiOutputRegressor(LGBMRegressor(
    objective='mae', n_estimators=1000, num_leaves=95, learning_rate=0.015, 
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, device='gpu'
)).fit(X, y)

del train_final, X, y
gc.collect()

# 2. Prepare the Test Set
print("1. Loading raw datasets...")
test_raw = pd.read_csv(test_data_path)

# Parse dates
test_date_parts = test_raw['date'].str.split('-', expand=True)
test_raw['year'] = test_date_parts[0].astype(int)
test_raw['month'] = test_date_parts[1].astype(int)
test_raw['day'] = test_date_parts[2].astype(int)

test_raw = test_raw.sort_values(['region_id', 'date']).reset_index(drop=True)
test_raw['evap_stress'] = test_raw['tmp_max'] * (100 - test_raw['humidity'])

# Climate Normals Baselines
test_raw = test_raw.merge(climate_normals_region_month, on=['region_id', 'month'], how='left')

# SAFE WINDOWS  
print("Engineering capped rolling windows...")

for w in windows:
    for col in weather_cols:
        test_raw[f'{col}_roll_mean_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).mean())
        test_raw[f'{col}_roll_max_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).max())
        if w >= 35:
            test_raw[f'{col}_roll_std_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).std())

# Anomalies
print("Engineering regional anomalies...")
for w in windows:
    for col in weather_cols:
        test_raw[f'{col}_anomaly_{w}'] = test_raw[f'{col}_roll_mean_{w}'] - test_raw[f'normal_{col}']

# Convert the region_id from a text string to a Pandas 'category' type
test_raw['region_id'] = test_raw['region_id'].astype('category')

# Merge this back into your main dataframe
test_raw = test_raw.merge(region_avg_score, on='region_id', how='left')
test_raw = test_raw.merge(month_region_avg_score, on=['region_id', 'month'], how='left')
test_final = test_raw.dropna().copy()

test_final = add_future_month_features(test_final, month_region_avg_score)
test_submission_rows = test_final.groupby('region_id').last().reset_index()
X_test = test_submission_rows[feature_names].copy()

# 3. Predict the future 5 weeks 
print("Generating Final 5-Week Forecasts...")

final_predictions = np.clip(master_model.predict(X_test), 0, 5)

# 3.a Find the nearest whole number for every single prediction
rounded_preds = np.floor(final_predictions)

# 3.b Calculate the absolute distance between the continuous prediction and the whole number
distance_to_int = np.abs(final_predictions - rounded_preds)

# 3.c If the distance is less than 0.20, snap to the whole number. Otherwise, keep the decimal.
final_predictions = np.where(distance_to_int < 0.20, rounded_preds, final_predictions)

# 4. Format the output exactly as Kaggle demands
submission = pd.DataFrame({
    'region_id': test_submission_rows['region_id'],
    'pred_week1': final_predictions[:, 0],
    'pred_week2': final_predictions[:, 1],
    'pred_week3': final_predictions[:, 2],
    'pred_week4': final_predictions[:, 3],
    'pred_week5': final_predictions[:, 4]
})

# Save the file
submission.to_csv('submission_last_5_week_window_region_monthly.csv', index=False)
print("\nSuccess  'submission_last_5_week_window_region_monthly.csv' is saved and ready to upload to Kaggle.")

Using device: cuda
1. Loading raw datasets...
Engineering capped rolling windows...
Engineering regional anomalies...

Total 7th-day valid labeled rows available for Model 1: 1737704
Training Final Master Model on all 1737704 rows...
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 40400
[LightGBM] [Info] Number of data points in the train set: 1737704, number of used features: 165
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 165 dense feature groups (278.41 MB) transferred to GPU in 0.175421 secs. 0 sparse feature groups
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 40400
[LightGBM] [Info] Number of data points in the train set: 1737704, number of used features: 165
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 165 dense feature groups (278.41 MB) transferred to GPU in 0.167442 secs. 0 sparse feature groups
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 40400
[LightGBM] [Info] Number of data points in the train set: 1737704, number of used features: 165
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Com

In [ ]:
# Increased n_estimator to 5000 and std started to be measured at w>=28, 0.25 rounding
from sklearn.multioutput import MultiOutputRegressor
from lightgbm import LGBMRegressor
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
import gc
import torch
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("1. Loading raw datasets...")
train_raw = pd.read_csv(train_data_path)

# Parse dates
train_date_parts = train_raw['date'].str.split('-', expand=True)
train_raw['year'] = train_date_parts[0].astype(int)
train_raw['month'] = train_date_parts[1].astype(int)
train_raw['day'] = train_date_parts[2].astype(int)

train_raw = train_raw.sort_values(['region_id', 'date']).reset_index(drop=True)

# Targets
targets = ['target_w1', 'target_w2', 'target_w3', 'target_w4', 'target_w5']
for i, t in enumerate(targets):
    train_raw[t] = train_raw.groupby('region_id')['score'].shift(-(i+1)*7)

train_raw['evap_stress'] = train_raw['tmp_max'] * (100 - train_raw['humidity'])

# Climate Normals Baselines
weather_cols = ['prec', 'humidity', 'wind', 'surf_tmp', 'evap_stress', 'surf_pre', 'dp_tmp', 'tmp']
climate_normals_region_month = train_raw.groupby(['region_id', 'month'])[weather_cols].mean().reset_index()
climate_normals_region_month = climate_normals_region_month.rename(columns={c: f'normal_{c}' for c in weather_cols})

train_raw = train_raw.merge(climate_normals_region_month, on=['region_id', 'month'], how='left')

# SAFE WINDOWS  
print("Engineering capped rolling windows...")
windows = [7, 14, 21, 28, 35]

for w in windows:
    for col in weather_cols:
        train_raw[f'{col}_roll_mean_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).mean())
        train_raw[f'{col}_roll_max_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).max())
        if w >= 28:
            train_raw[f'{col}_roll_std_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).std())

# Anomalies
print("Engineering regional anomalies...")
for w in windows:
    for col in weather_cols:
        train_raw[f'{col}_anomaly_{w}'] = train_raw[f'{col}_roll_mean_{w}'] - train_raw[f'normal_{col}']
        
train_final = train_raw.dropna().copy()

# Convert the region_id from a text string to a Pandas 'category' type
train_final['region_id'] = train_final['region_id'].astype('category')

# Calculate the historical average score for each region
region_avg_score = train_final.groupby('region_id')['score'].mean().reset_index().rename(columns={'score': 'historical_region_score'})

# Merge this back into your main dataframe
train_final = train_final.merge(region_avg_score, on='region_id', how='left')

# Monthly region average score
month_region_avg_score = train_final.groupby(['region_id', 'month'])['score'].mean().reset_index().rename(columns={'score': 'month_region_score'})
# Merge this back into your main dataframe
train_final = train_final.merge(month_region_avg_score, on=['region_id', 'month'], how='left')

print(f"\nTotal 7th-day valid labeled rows available for Model 1: {len(train_final)}")

MONTH_LEN = {
    1: 31, 2: 28, 3: 31, 4: 30, 5: 31, 6: 30,
    7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31
}

def is_leap_year(y):
    return (y % 4 == 0 and y % 100  = 0) or (y % 400 == 0)

def add_days_get_future_month(year, month, day, add_days):
    y, m, d = int(year), int(month), int(day)
    
    while add_days > 0:
        month_len = 29 if (m == 2 and is_leap_year(y)) else MONTH_LEN[m]
        days_left_in_month = month_len - d
        
        if add_days <= days_left_in_month:
            d += add_days
            add_days = 0
        else:
            add_days -= (days_left_in_month + 1)
            d = 1
            m += 1
            if m > 12:
                m = 1
                y += 1
    
    return m
    
def add_future_month_features(df, region_avg_score, horizons=(1, 2, 3, 4, 5)):
    df = df.copy()
    
    for h in horizons:
        add_days = 7 * h
        df[f'future_month_w{h}'] = [
            add_days_get_future_month(y, m, d, add_days)
            for y, m, d in zip(df['year'], df['month'], df['day'])
        ]
        
        tmp = df[['region_id', f'future_month_w{h}']].rename(
            columns={f'future_month_w{h}': 'month'}
        )
        
        tmp = tmp.merge(
            region_avg_score,
            on=['region_id', 'month'],
            how='left'
        )

        df[f'future_region_month_score_w{h}'] = tmp['month_region_score'].values

    return df

train_final = add_future_month_features(train_final, month_region_avg_score)

feature_names = [c for c in train_final.columns if c not in ['region_id', 'date', 'score', 'year'] + targets]

X = train_final[feature_names]
y = train_final[targets].values

# 1.  use the FULL dataset (X, y, and global_sample_weights)
print(f"Training Final Master Model on all {len(X)} rows...")

master_model = MultiOutputRegressor(LGBMRegressor(
    objective='mae', n_estimators=5000, num_leaves=63, learning_rate=0.01, 
    subsample=0.6, subsample_freq = 1 ,colsample_bytree=0.6, random_state=42, n_jobs=-1, lambda_l1 = '2', lambda_l2 = '20',device='gpu'
)).fit(X, y)

del train_final, X, y
gc.collect()

# 2. Prepare the Test Set
print("1. Loading raw datasets...")
test_raw = pd.read_csv(test_data_path)

# Parse dates
test_date_parts = test_raw['date'].str.split('-', expand=True)
test_raw['year'] = test_date_parts[0].astype(int)
test_raw['month'] = test_date_parts[1].astype(int)
test_raw['day'] = test_date_parts[2].astype(int)

test_raw = test_raw.sort_values(['region_id', 'date']).reset_index(drop=True)
test_raw['evap_stress'] = test_raw['tmp_max'] * (100 - test_raw['humidity'])

# Climate Normals Baselines
test_raw = test_raw.merge(climate_normals_region_month, on=['region_id', 'month'], how='left')

# SAFE WINDOWS  
print("Engineering capped rolling windows...")

for w in windows:
    for col in weather_cols:
        test_raw[f'{col}_roll_mean_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).mean())
        test_raw[f'{col}_roll_max_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).max())
        if w >= 28:
            test_raw[f'{col}_roll_std_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).std())

# Anomalies
print("Engineering regional anomalies...")
for w in windows:
    for col in weather_cols:
        test_raw[f'{col}_anomaly_{w}'] = test_raw[f'{col}_roll_mean_{w}'] - test_raw[f'normal_{col}']

# Convert the region_id from a text string to a Pandas 'category' type
test_raw['region_id'] = test_raw['region_id'].astype('category')

# Merge this back into your main dataframe
test_raw = test_raw.merge(region_avg_score, on='region_id', how='left')
test_raw = test_raw.merge(month_region_avg_score, on=['region_id', 'month'], how='left')
test_final = test_raw.dropna().copy()

test_final = add_future_month_features(test_final, month_region_avg_score)
test_submission_rows = test_final.groupby('region_id').last().reset_index()
X_test = test_submission_rows[feature_names].copy()

# 3. Predict the future 5 weeks 
print("Generating Final 5-Week Forecasts...")

final_predictions = np.clip(master_model.predict(X_test), 0, 5)

# 3.a Find the nearest whole number for every single prediction
rounded_preds = np.floor(final_predictions)

# 3.b Calculate the absolute distance between the continuous prediction and the whole number
distance_to_int = np.abs(final_predictions - rounded_preds)

# 3.c If the distance is less than 0.20, snap to the whole number. Otherwise, keep the decimal.
final_predictions = np.where(distance_to_int < 0.25, rounded_preds, final_predictions)

# 4. Format the output exactly as Kaggle demands
submission = pd.DataFrame({
    'region_id': test_submission_rows['region_id'],
    'pred_week1': final_predictions[:, 0],
    'pred_week2': final_predictions[:, 1],
    'pred_week3': final_predictions[:, 2],
    'pred_week4': final_predictions[:, 3],
    'pred_week5': final_predictions[:, 4]
})

# Save the file
submission.to_csv('submission_last_5_week_window_region_monthly.csv', index=False)
print("\nSuccess  'submission_last_5_week_window_region_monthly.csv' is saved and ready to upload to Kaggle.")

Using device: cuda
1. Loading raw datasets...
Engineering capped rolling windows...
Engineering regional anomalies...

Total 7th-day valid labeled rows available for Model 1: 1737704
Training Final Master Model on all 1737704 rows...
[LightGBM] [Warning] lambda_l2 is set=20, reg_lambda=0.0 will be ignored. Current value: lambda_l2=20
[LightGBM] [Warning] lambda_l1 is set=2, reg_alpha=0.0 will be ignored. Current value: lambda_l1=2
[LightGBM] [Warning] lambda_l2 is set=20, reg_lambda=0.0 will be ignored. Current value: lambda_l2=20
[LightGBM] [Warning] lambda_l1 is set=2, reg_alpha=0.0 will be ignored. Current value: lambda_l1=2
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 42440
[LightGBM] [Info] Number of data points in the train set: 1737704, number of used features: 173
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 173 dense feature groups (291.67 MB) transferred to GPU in 0.177760 secs. 0 sparse feature groups
[LightGBM] [Warning] lambda_l2 is set=20, reg_lambda=0.0 will be ignored. Current value: lambda_l2=20
[LightGBM] [Warning] lambda_l1 is set=2, reg_alpha=0.0 will be ignored. Current value: lambda_l1=2
[LightGBM] [Warning] lambda_l2 is set=20, reg_lambda=0.0 will be ignored. Current value: lambda_l2=20
[LightGBM] [Warning] lambda_l1 is set=2, reg_alpha=0.0 will be ignored. Current value: lambda_l1=2
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 42440
[LightGBM] [Info] Number of data points in the train set: 1737704, number of used features: 173
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histog

In [ ]:
# Best Performance in the Public Leaderboard
from sklearn.multioutput import MultiOutputRegressor
from lightgbm import LGBMRegressor
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
import gc
import torch
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("1. Loading raw datasets...")
train_raw = pd.read_csv(train_data_path)

# Parse dates
train_date_parts = train_raw['date'].str.split('-', expand=True)
train_raw['year'] = train_date_parts[0].astype(int)
train_raw['month'] = train_date_parts[1].astype(int)
train_raw['day'] = train_date_parts[2].astype(int)

train_raw = train_raw.sort_values(['region_id', 'date']).reset_index(drop=True)

# Targets
targets = ['target_w1', 'target_w2', 'target_w3', 'target_w4', 'target_w5']
for i, t in enumerate(targets):
    train_raw[t] = train_raw.groupby('region_id')['score'].shift(-(i+1)*7)

train_raw['evap_stress'] = train_raw['tmp_max'] * (100 - train_raw['humidity'])
train_raw['evap_demand'] = train_raw['tmp_max'] / (train_raw['prec'] * 0.01)

# Climate Normals Baselines
weather_cols = ['prec', 'humidity', 'wind', 'surf_tmp', 'evap_stress', 'surf_pre', 'dp_tmp', 'tmp', 'evap_demand']
climate_normals_region_month = train_raw.groupby(['region_id', 'month'])[weather_cols].mean().reset_index()
climate_normals_region_month = climate_normals_region_month.rename(columns={c: f'normal_{c}' for c in weather_cols})

train_raw = train_raw.merge(climate_normals_region_month, on=['region_id', 'month'], how='left')

# SAFE WINDOWS  
print("Engineering capped rolling windows...")
windows = [7, 14, 21, 28, 42]

for w in windows:
    for col in weather_cols:
        train_raw[f'{col}_roll_mean_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w,min_periods=1).mean())
        train_raw[f'{col}_roll_max_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w,min_periods=1).max())
        if w >= 28:
            train_raw[f'{col}_roll_std_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w,min_periods=1).std())

# Anomalies
print("Engineering regional anomalies...")
for w in windows:
    for col in weather_cols:
        train_raw[f'{col}_anomaly_{w}'] = train_raw[f'{col}_roll_mean_{w}'] - train_raw[f'normal_{col}']
        
train_final = train_raw.dropna().copy()

# Convert the region_id from a text string to a Pandas 'category' type
train_final['region_id'] = train_final['region_id'].astype('category')

# Calculate the historical average score for each region
region_avg_score = train_final.groupby('region_id')['score'].mean().reset_index().rename(columns={'score': 'historical_region_score'})

# Merge this back into your main dataframe
train_final = train_final.merge(region_avg_score, on='region_id', how='left')

# Monthly region average score
month_region_avg_score = train_final.groupby(['region_id', 'month'])['score'].mean().reset_index().rename(columns={'score': 'month_region_score'})
# Merge this back into your main dataframe
train_final = train_final.merge(month_region_avg_score, on=['region_id', 'month'], how='left')

print(f"\nTotal 7th-day valid labeled rows available for Model 1: {len(train_final)}")

MONTH_LEN = {
    1: 31, 2: 28, 3: 31, 4: 30, 5: 31, 6: 30,
    7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31
}

def is_leap_year(y):
    return (y % 4 == 0 and y % 100  = 0) or (y % 400 == 0)

def add_days_get_future_month(year, month, day, add_days):
    y, m, d = int(year), int(month), int(day)
    
    while add_days > 0:
        month_len = 29 if (m == 2 and is_leap_year(y)) else MONTH_LEN[m]
        days_left_in_month = month_len - d
        
        if add_days <= days_left_in_month:
            d += add_days
            add_days = 0
        else:
            add_days -= (days_left_in_month + 1)
            d = 1
            m += 1
            if m > 12:
                m = 1
                y += 1
    
    return m
    
def add_future_month_features(df, region_avg_score, horizons=(1, 2, 3, 4, 5)):
    df = df.copy()
    
    for h in horizons:
        add_days = 7 * h
        df[f'future_month_w{h}'] = [
            add_days_get_future_month(y, m, d, add_days)
            for y, m, d in zip(df['year'], df['month'], df['day'])
        ]
        
        tmp = df[['region_id', f'future_month_w{h}']].rename(
            columns={f'future_month_w{h}': 'month'}
        )
        
        tmp = tmp.merge(
            region_avg_score,
            on=['region_id', 'month'],
            how='left'
        )

        df[f'future_region_month_score_w{h}'] = tmp['month_region_score'].values

    return df

train_final = add_future_month_features(train_final, month_region_avg_score)

feature_names = [c for c in train_final.columns if c not in ['region_id', 'date', 'score', 'year'] + targets]

X = train_final[feature_names]
y = train_final[targets].values

# 1. Use the FULL dataset (X, y, and global_sample_weights)
print(f"Training Final Master Model on all {len(X)} rows...")

master_model = MultiOutputRegressor(LGBMRegressor(
    objective='mae', n_estimators=5000, num_leaves=63, learning_rate=0.01, 
    subsample=0.6, subsample_freq = 1 ,colsample_bytree=0.6, random_state=42, n_jobs=-1, lambda_l1 = '2', lambda_l2 = '20',device='gpu'
)).fit(X, y)

del train_final, X, y
gc.collect()

# 2. Prepare the Test Set 
print("1. Loading raw datasets...")
test_raw = pd.read_csv(test_data_path)

# Parse dates
test_date_parts = test_raw['date'].str.split('-', expand=True)
test_raw['year'] = test_date_parts[0].astype(int)
test_raw['month'] = test_date_parts[1].astype(int)
test_raw['day'] = test_date_parts[2].astype(int)

test_raw = test_raw.sort_values(['region_id', 'date']).reset_index(drop=True)
test_raw['evap_stress'] = test_raw['tmp_max'] * (100 - test_raw['humidity'])
test_raw['evap_demand'] = test_raw['tmp_max'] / (test_raw['prec'] * 0.01)

# Climate Normals Baselines
test_raw = test_raw.merge(climate_normals_region_month, on=['region_id', 'month'], how='left')

# SAFE WINDOWS  
print("Engineering capped rolling windows...")

for w in windows:
    for col in weather_cols:
        test_raw[f'{col}_roll_mean_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w, min_periods=1).mean())
        test_raw[f'{col}_roll_max_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w, min_periods=1).max())
        if w >= 28:
            test_raw[f'{col}_roll_std_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w, min_periods=1).std())

# Anomalies
print("Engineering regional anomalies...")
for w in windows:
    for col in weather_cols:
        test_raw[f'{col}_anomaly_{w}'] = test_raw[f'{col}_roll_mean_{w}'] - test_raw[f'normal_{col}']

# Convert the region_id from a text string to a Pandas 'category' type
test_raw['region_id'] = test_raw['region_id'].astype('category')

# Merge this back into your main dataframe
test_raw = test_raw.merge(region_avg_score, on='region_id', how='left')
test_raw = test_raw.merge(month_region_avg_score, on=['region_id', 'month'], how='left')
test_final = test_raw.dropna().copy()

test_final = add_future_month_features(test_final, month_region_avg_score)
test_submission_rows = test_final.groupby('region_id').last().reset_index()
X_test = test_submission_rows[feature_names].copy()

# 3. Predict the future 5 weeks 
print("Generating Final 5-Week Forecasts...")

final_predictions = np.clip(master_model.predict(X_test), 0, 5)

# 3.a Find the nearest whole number for every single prediction
rounded_preds = np.floor(final_predictions)

# 3.b Calculate the absolute distance between the continuous prediction and the whole number
distance_to_int = np.abs(final_predictions - rounded_preds)

# 3.c If the distance is less than 0.20, snap to the whole number. Otherwise, keep the decimal.
final_predictions = np.where(distance_to_int < 0.20, rounded_preds, final_predictions)

# 4. Format the output exactly as Kaggle demands
submission = pd.DataFrame({
    'region_id': test_submission_rows['region_id'],
    'pred_week1': final_predictions[:, 0],
    'pred_week2': final_predictions[:, 1],
    'pred_week3': final_predictions[:, 2],
    'pred_week4': final_predictions[:, 3],
    'pred_week5': final_predictions[:, 4]
})

# Save the file
submission.to_csv('submission_last_4&8_weeks_window_region_monthly.csv', index=False)
print("\nSuccess  'submission_last_4&8_weeks_window_region_monthly.csv' is saved and ready to upload to Kaggle.")
# Typo in the naming convention, should be 4&6 weeks

# above is the latest, below is trial #

In [ ]:
from sklearn.multioutput import MultiOutputRegressor
from lightgbm import LGBMRegressor
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
import gc
import torch
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

print("1. Loading raw datasets...")
train_raw = pd.read_csv(train_data_path)

# Parse dates
train_date_parts = train_raw['date'].str.split('-', expand=True)
train_raw['year'] = train_date_parts[0].astype(int)
train_raw['month'] = train_date_parts[1].astype(int)
train_raw['day'] = train_date_parts[2].astype(int)

train_raw = train_raw.sort_values(['region_id', 'date']).reset_index(drop=True)

# Targets
targets = ['target_w1', 'target_w2', 'target_w3', 'target_w4', 'target_w5']
for i, t in enumerate(targets):
    train_raw[t] = train_raw.groupby('region_id')['score'].shift(-(i+1)*7)
    
train_raw['evap_stress'] = train_raw['tmp_max'] * (100 - train_raw['humidity'])

# Climate Normals Baselines
weather_cols = ['prec', 'humidity', 'wind', 'surf_tmp', 'evap_stress', 'surf_pre', 'dp_tmp', 'tmp']
climate_normals_region_month = train_raw.groupby(['region_id', 'month'])[weather_cols].mean().reset_index()
climate_normals_region_month = climate_normals_region_month.rename(columns={c: f'normal_{c}' for c in weather_cols})

train_raw = train_raw.merge(climate_normals_region_month, on=['region_id', 'month'], how='left')

# SAFE WINDOWS  
print("Engineering capped rolling windows...")
windows = [7, 14, 21, 28, 35]

for w in windows:
    for col in weather_cols:
        train_raw[f'{col}_roll_mean_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).mean())
        train_raw[f'{col}_roll_max_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).max())
        if w >= 28:
            train_raw[f'{col}_roll_std_{w}'] = train_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).std())

# Anomalies
print("Engineering regional anomalies...")
for w in windows:
    for col in weather_cols:
        train_raw[f'{col}_anomaly_{w}'] = train_raw[f'{col}_roll_mean_{w}'] - train_raw[f'normal_{col}']
        
train_final = train_raw.dropna().copy()

# Convert the region_id from a text string to a Pandas 'category' type
train_final['region_id'] = train_final['region_id'].astype('category')

# Calculate the historical average score for each region
region_avg_score = train_final.groupby('region_id')['score'].mean().reset_index().rename(columns={'score': 'historical_region_score'})

# Merge this back into your main dataframe
train_final = train_final.merge(region_avg_score, on='region_id', how='left')

# Monthly region average score
month_region_avg_score = train_final.groupby(['region_id', 'month'])['score'].mean().reset_index().rename(columns={'score': 'month_region_score'})

# Merge this back into your main dataframe
train_final = train_final.merge(month_region_avg_score, on=['region_id', 'month'], how='left')

print(f"\nTotal 7th-day valid labeled rows available for Model 1: {len(train_final)}")

MONTH_LEN = {
    1: 31, 2: 28, 3: 31, 4: 30, 5: 31, 6: 30,
    7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31
}

def is_leap_year(y):
    return (y % 4 == 0 and y % 100  = 0) or (y % 400 == 0)

def add_days_get_future_month(year, month, day, add_days):
    y, m, d = int(year), int(month), int(day)
    
    while add_days > 0:
        month_len = 29 if (m == 2 and is_leap_year(y)) else MONTH_LEN[m]
        days_left_in_month = month_len - d
        
        if add_days <= days_left_in_month:
            d += add_days
            add_days = 0
        else:
            add_days -= (days_left_in_month + 1)
            d = 1
            m += 1
            if m > 12:
                m = 1
                y += 1
    
    return m
    
def add_future_month_features(df, region_avg_score, horizons=(1, 2, 3, 4, 5)):
    df = df.copy()
    
    for h in horizons:
        add_days = 7 * h
        df[f'future_month_w{h}'] = [
            add_days_get_future_month(y, m, d, add_days)
            for y, m, d in zip(df['year'], df['month'], df['day'])
        ]
        
        tmp = df[['region_id', f'future_month_w{h}']].rename(
            columns={f'future_month_w{h}': 'month'}
        )
        
        tmp = tmp.merge(
            region_avg_score,
            on=['region_id', 'month'],
            how='left'
        )

        df[f'future_region_month_score_w{h}'] = tmp['month_region_score'].values

    return df

train_final = add_future_month_features(train_final, month_region_avg_score)

feature_names = [c for c in train_final.columns if c not in ['region_id', 'date', 'score', 'year'] + targets]

X = train_final[feature_names]
y = train_final[targets].values

# 1.  use the FULL dataset (X, y, and global_sample_weights)
print(f"Training Final Master Model on all {len(X)} rows...")

master_model = MultiOutputRegressor(LGBMRegressor(
    objective='mae', n_estimators=6000, num_leaves=63, learning_rate=0.01, 
    subsample=0.7, subsample_freq = 1 ,colsample_bytree=0.7, random_state=42, n_jobs=-1, lambda_l1 = '2', lambda_l2 = '20',device='gpu'
)).fit(X, y)

del train_final, X, y
gc.collect()

# 2. Prepare the Test Set
print("1. Loading raw datasets...")
test_raw = pd.read_csv(test_data_path)

# Parse dates
test_date_parts = test_raw['date'].str.split('-', expand=True)
test_raw['year'] = test_date_parts[0].astype(int)
test_raw['month'] = test_date_parts[1].astype(int)
test_raw['day'] = test_date_parts[2].astype(int)

test_raw = test_raw.sort_values(['region_id', 'date']).reset_index(drop=True)
test_raw['evap_stress'] = test_raw['tmp_max'] * (100 - test_raw['humidity'])

# Climate Normals Baselines
test_raw = test_raw.merge(climate_normals_region_month, on=['region_id', 'month'], how='left')

# SAFE WINDOWS  
print("Engineering capped rolling windows...")

for w in windows:
    for col in weather_cols:
        test_raw[f'{col}_roll_mean_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).mean())
        test_raw[f'{col}_roll_max_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).max())
        if w >= 28:
            test_raw[f'{col}_roll_std_{w}'] = test_raw.groupby('region_id')[col].transform(lambda x: x.rolling(w).std())

# Anomalies
print("Engineering regional anomalies...")
for w in windows:
    for col in weather_cols:
        test_raw[f'{col}_anomaly_{w}'] = test_raw[f'{col}_roll_mean_{w}'] - test_raw[f'normal_{col}']

# Convert the region_id from a text string to a Pandas 'category' type
test_raw['region_id'] = test_raw['region_id'].astype('category')

# Merge this back into your main dataframe
test_raw = test_raw.merge(region_avg_score, on='region_id', how='left')
test_raw = test_raw.merge(month_region_avg_score, on=['region_id', 'month'], how='left')
test_final = test_raw.dropna().copy()

test_final = add_future_month_features(test_final, month_region_avg_score)
test_submission_rows = test_final.groupby('region_id').last().reset_index()
X_test = test_submission_rows[feature_names].copy()

# 3. Predict the future 5 weeks 
print("Generating Final 5-Week Forecasts...")

final_predictions = np.clip(master_model.predict(X_test), 0, 5)

# 1. Standard 0.25 Snap for the whole dataset
rounded_preds = np.floor(final_predictions)
distance_to_int = np.abs(final_predictions - rounded_preds)
final_predictions = np.where(distance_to_int < 0.10, rounded_preds, final_predictions)

# 2. Aggressive Zero-Snap (If it's under 0.35, it's just noisy healthy soil. Force it to 0.0)
final_predictions = np.where(final_predictions < 0.25, 0.0, final_predictions)

# 3. Re-clip just to be completely safe
final_predictions = np.clip(final_predictions, 0, 5)

# 4. Format the output exactly as Kaggle demands
submission = pd.DataFrame({
    'region_id': test_submission_rows['region_id'],
    'pred_week1': final_predictions[:, 0],
    'pred_week2': final_predictions[:, 1],
    'pred_week3': final_predictions[:, 2],
    'pred_week4': final_predictions[:, 3],
    'pred_week5': final_predictions[:, 4]
})

# Save the file
submission.to_csv('submission_last_5_week_window_region_monthly_10_rounded.csv', index=False)
print("\nSuccess  'submission_last_5_week_window_region_monthly_10_rounded.csv' is saved and ready to upload to Kaggle.")

Using device: cuda
1. Loading raw datasets...
Engineering capped rolling windows...
Engineering regional anomalies...

Total 7th-day valid labeled rows available for Model 1: 1737704
Training Final Master Model on all 1737704 rows...
[LightGBM] [Warning] lambda_l2 is set=20, reg_lambda=0.0 will be ignored. Current value: lambda_l2=20
[LightGBM] [Warning] lambda_l1 is set=2, reg_alpha=0.0 will be ignored. Current value: lambda_l1=2
[LightGBM] [Warning] lambda_l2 is set=20, reg_lambda=0.0 will be ignored. Current value: lambda_l2=20
[LightGBM] [Warning] lambda_l1 is set=2, reg_alpha=0.0 will be ignored. Current value: lambda_l1=2
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 42440
[LightGBM] [Info] Number of data points in the train set: 1737704, number of used features: 173
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 173 dense feature groups (291.67 MB) transferred to GPU in 0.181817 secs. 0 sparse feature groups
